<a target="_blank" href="https://colab.research.google.com/github/PacktPublishing/Building-Agentic-AI-Systems/blob/main/Chapter_02.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Chapter 2: Principles of Agentic Systems
---

### Implementing Algorithm 1: Travel Booking Assistant Algorithm with Agency and Autonomy

This is a simple Python implementation of Algorithm 1 in Chapter 2.

In [2]:
# 1. Clone the repository
!git clone https://github.com/PacktPublishing/Building-Agentic-AI-Systems.git

# 2. Move into the Chapter 2 directory where travel_provider.py lives
import sys
sys.path.append('/content/Building-Agentic-AI-Systems/Chapter02')

Cloning into 'Building-Agentic-AI-Systems'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 147 (delta 12), reused 11 (delta 11), pack-reused 132 (from 2)
Receiving objects: 100% (147/147), 412.60 KiB | 8.78 MiB/s, done.
Resolving deltas: 100% (71/71), done.


In [4]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.2 MB/s eta 0:00:00


In [5]:
from travel_provider import travel_provider
from typing import List, Dict, Any

## TravelProvider

In [9]:
# travel_provider.py
import json
import os
from faker import Faker
from faker.providers import BaseProvider
import random

# Load supported locations from the JSON file
def load_supported_locations():
    # Check if __file__ is defined (normal script execution)
    if '__file__' in globals():
        base_path = os.path.dirname(__file__)
    else:
        # Fallback for Google Colab/Jupyter Notebooks
        base_path = '/content/Building-Agentic-AI-Systems/Chapter02'

    json_file = os.path.join(base_path, 'supported_locations.json')

    with open(json_file, 'r') as f:
        return json.load(f)

supported_locations = load_supported_locations()
print(supported_locations)


{'airports': ['LAX', 'JFK', 'ORD', 'ATL', 'DFW', 'DEN', 'SEA', 'SAN'], 'hotel_cities': ['New York', 'Los Angeles', 'Chicago', 'Miami', 'San Francisco', 'Seattle', 'San Diego']}


In [10]:


class TravelProvider(BaseProvider):
    def __init__(self, faker_instance):
        self.fake = faker_instance  # Use the provided Faker instance

    def flight_lookup(self, departure_city, destination_city, num_options=3):
        """
        Generate a list of flight options between the specified departure and destination cities.
        Validates if the cities are in the supported airports.
        Includes pricing for Economy, Economy+, and Business classes.
        """
        if departure_city not in supported_locations['airports']:
            return {"error": f"Unsupported departure city: {departure_city}. Supported airports are {supported_locations['airports']}"}
        if destination_city not in supported_locations['airports']:
            return {"error": f"Unsupported destination city: {destination_city}. Supported airports are {supported_locations['airports']}"}
        if departure_city == destination_city:
            return {"error": "Departure and destination cities cannot be the same."}

        flights = []
        for _ in range(num_options):
            airline = random.choice(['Delta', 'United', 'Southwest', 'JetBlue', 'American Airlines'])
            flight_number = f"{random.choice(['DL', 'UA', 'SW', 'JB', 'AA'])}{random.randint(100, 9999)}"
            departure_time = self.fake.date_time_between(start_date="now", end_date="+30d")
            arrival_time = self.fake.date_time_between(start_date=departure_time, end_date="+30d")

            # Generate random prices for each class
            economy_price = round(random.uniform(100, 300), 2)
            # economy_plus_price = round(random.uniform(200, 400), 2)
            # business_price = round(random.uniform(500, 1000), 2)

            flights.append({
                'airline': airline,
                'departure_airport': departure_city,
                'destination_airport': destination_city,
                'flight_number': flight_number,
                'departure_time': departure_time,
                'arrival_time': arrival_time,
                'price':  economy_price
            })
        return {"status_code": 200, "flight_options":flights}

    def hotel_lookup(self, city, num_options=3):
        """
        Generate a list of hotel options in the specified city.
        Validates if the city is in the supported hotel cities.
        """
        if city not in supported_locations['hotel_cities']:
            return {"error": f"Unsupported city: {city}. Supported cities are {supported_locations['hotel_cities']}"}

        hotels = []
        for _ in range(num_options):
            hotel_name = random.choice(['Hilton', 'Marriott', 'Hyatt', 'Holiday Inn', 'Sheraton'])
            check_in = self.fake.date_time_between(start_date="now", end_date="+30d")
            check_out = self.fake.date_time_between(start_date=check_in, end_date="+35d")
            price_per_night = random.uniform(100, 500)
            total_price = price_per_night * random.randint(1, 7)  # 1 to 7 nights

            hotels.append({
                'hotel_name': hotel_name,
                'city': city,
                'check_in': check_in,
                'check_out': check_out,
                'price_per_night': round(price_per_night, 2),
                'total_price': round(total_price, 2)
            })
        return hotels

# Initialize Faker globally
fake = Faker()

# Initialize TravelProvider with the Faker instance
travel_provider = TravelProvider(fake)

In [13]:
fake

In [15]:
fake.date_time_between(start_date="now", end_date="+30d")

datetime.datetime(2026, 6, 18, 7, 4, 45, 424721)

In [ ]:
fake.

In [14]:
travel_provider

In [6]:
from travel_provider import travel_provider
from typing import List, Dict, Any

class TravelAgent:
    def __init__(self, name: str):
        self.name = name
        self.goals: List[str] = []
        self.knowledge_base: Dict[str, Any] = {}

    def set_goal(self, goal: str):
        """Agency: Defining objectives"""
        self.goals.append(goal)
        print(f"Goal set: {goal}")

    def update_knowledge(self, departure: str, destination: str):
        """Agency: Acquiring information from an API, and scoring"""
        # Simulating API call to get flight options
        response = travel_provider.flight_lookup(departure, destination)
        if response['status_code'] == 200:
            flight_options = response['flight_options']
            # Simple scoring based on price (lower is better)
            scored_options = [
                {**flight, 'score': 1000 / flight['price']}
                for flight in flight_options
            ]
            self.knowledge_base['flight_options'] = scored_options
            print(f"Knowledge updated with {len(scored_options)} flight options")
        else:
            print("Failed to fetch flight information")

    def make_decision(self) -> Dict[str, Any]:
        """Autonomy: Independent decision-making"""
        if 'flight_options' not in self.knowledge_base:
            raise ValueError("No flight options available for decision-making")
        best_option = max(self.knowledge_base['flight_options'], key=lambda x: x['score'])
        print(f"Decision made: Selected flight {best_option['airline']}")
        return best_option

    def book_travel(self, departure: str, destination: str):
        """Agency: Execute action on behalf of user"""
        print(f"Agent {self.name} is booking travel from {departure} to {destination}")

        self.set_goal(f"Book flight from {departure} to {destination}")
        self.update_knowledge(departure, destination)

        try:
            best_flight = self.make_decision()
            # Simulating booking process
            booking_confirmation = f"BOOK-{best_flight['airline']}-{self.name.upper()}"
            self.knowledge_base['booking_confirmation'] = booking_confirmation
            print(f"Booking confirmed: {booking_confirmation}")
        except Exception as e:
            print(f"Booking failed: {str(e)}")

        return self

# Usage example
if __name__ == "__main__":
    agent = TravelAgent("TripPlanner")
    agent.book_travel("SAN", "SEA")
    print("\n----------- Final Agent State: -----------")
    print(f"Name: {agent.name}")
    print(f"Goals: {agent.goals}")
    print(f"Knowledge Base: {agent.knowledge_base}")
    if 'booking_confirmation' in agent.knowledge_base:
        print(f"Booking Confirmation: {agent.knowledge_base['booking_confirmation']}")

Agent TripPlanner is booking travel from SAN to SEA
Goal set: Book flight from SAN to SEA
Knowledge updated with 3 flight options
Decision made: Selected flight Delta
Booking confirmed: BOOK-Delta-TRIPPLANNER

----------- Final Agent State: -----------
Name: TripPlanner
Goals: ['Book flight from SAN to SEA']
Knowledge Base: {'flight_options': [{'airline': 'United', 'departure_airport': 'SAN', 'destination_airport': 'SEA', 'flight_number': 'AA792', 'departure_time': datetime.datetime(2026, 7, 7, 7, 8, 10, 912019), 'arrival_time': datetime.datetime(2026, 7, 8, 1, 13, 0, 318175), 'price': 270.25, 'score': 3.700277520814061}, {'airline': 'United', 'departure_airport': 'SAN', 'destination_airport': 'SEA', 'flight_number': 'SW2642', 'departure_time': datetime.datetime(2026, 6, 22, 13, 23, 52, 609190), 'arrival_time': datetime.datetime(2026, 7, 1, 7, 50, 35, 272090), 'price': 266.67, 'score': 3.7499531255859297}, {'airline': 'Delta', 'departure_airport': 'SAN', 'destination_airport': 'SEA', '